# DPE Head Reweighting: End-to-End Experiment

This general Jupyter notebook prepares Waterbirds, fine-tunes a shared ERM backbone, extracts raw features, trains the DPE heads, selects aggregation weights and evaluates the held-out test split. It delegates numerical methods to repository modules.

**Prerequisites:** a writable repository checkout, Python 3.10 or later, a matching PyTorch/torchvision installation and sufficient storage for the dataset and checkpoints. Initial dataset and ImageNet-weight downloads require network access. A GPU is recommended for the 100-epoch backbone run; CPU execution is supported but slow.

The default track uses standard ERM. The enhanced track uses ERM* with training augmentation. Both start from ImageNet-pretrained ResNet-50, rather than random initialization. Existing completed shared ERM runs can be supplied through `BACKBONE_DIR`; no precomputed features or manually written provenance files are required.

Validation A fits/selects head epochs, B estimates reliability, and C selects aggregation. Test predictions are evaluated only after selection is saved. See the [method specification](../src/methods/dpe_head_reweighting/README.md). Real Waterbirds performance and GPU parity remain unverified.

In [ ]:
# Environment initialization. Set REPO_PATH if automatic discovery is insufficient.
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

REPO_PATH = None  # Optional explicit path to a writable repository checkout.

if REPO_PATH is not None:
    REPO = Path(REPO_PATH).expanduser().resolve()
else:
    working_dir = Path.cwd().resolve()
    candidates = [working_dir, *working_dir.parents]
    candidates += [working_dir / "waterbirds-shortcut-mitigation"]
    REPO = next((path for path in candidates
                 if (path / "scripts/dpe_head_reweighting.py").is_file()), None)
if REPO is None or not (REPO / "scripts/dpe_head_reweighting.py").is_file():
    raise FileNotFoundError("Set REPO_PATH to a writable waterbirds-shortcut-mitigation checkout.")

# Install missing dependencies only. Use a matching torch/torchvision pair for the runtime.
# Network access is also needed for dataset and initial ImageNet-weight downloads.
missing = [package for module, package in
           [("numpy", "numpy>=1.26,<3"), ("torch", "torch>=2.6,<3"),
            ("torchvision", "torchvision>=0.21,<1"), ("pandas", "pandas>=2.2,<3"),
            ("PIL", "Pillow>=10.3,<13"), ("yaml", "PyYAML>=6,<7"),
            ("matplotlib", "matplotlib"), ("tqdm", "tqdm")]
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Repository:", REPO)
print("Python:", sys.version.split()[0])
# Existing checkouts are used as supplied; no revision changes are performed.
try:
    print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
except (OSError, subprocess.CalledProcessError):
    print("Git commit unavailable; implementation file hashes will still be recorded.")

## 1. Configuration

Set `DATA_DIR`, the track and seed. `DOWNLOAD_DATA` prepares the dataset if metadata is absent. Existing original or project-prepared dataset layouts are accepted. `BACKBONE_DIR` may identify a completed run from the shared ERM script with the matching recipe and seed, or a new directory for training. `INPUT_DIR` stores the generated feature bundle; `OUTPUT_DIR` stores reweighting results. All output locations must be writable.

Completed stages are reused after input checks. A changed recipe requires a new backbone directory; a changed feature source requires a new input directory; a changed H2 configuration requires a new run name. The shared trainer currently cannot resume partial backbone training: preserve an interrupted run and choose a fresh `BACKBONE_DIR`.

In [ ]:
import json
import torch
import yaml

TRACK = "standard"  # "standard" (ERM) or "enhanced" (ERM*)
SEED = 0            # 0, 1 or 2
RUN_NAME = f"h2-{TRACK}-seed{SEED}"
DATA_DIR = (REPO / "data/waterbirds").resolve()  # Existing dataset may be read-only.
DOWNLOAD_DATA = True  # Download when metadata is absent; False requires existing data.
BACKBONE_DIR = (REPO / f"checkpoints/waterbirds_stage0_{TRACK}_erm" / f"seed{SEED}").resolve()
INPUT_DIR = (REPO / "outputs/dpe_features" / TRACK / f"seed{SEED}").resolve()
OUTPUT_DIR = (REPO / "outputs/dpe_head_reweighting_runs").resolve()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORKERS = 2

from src.methods.dpe_head_reweighting.stage0 import (
    backbone_config, export_inputs, train_or_reuse, validate_dataset,
)
from src.methods.dpe_head_reweighting.experiment import (
    config, context, digest, get_ensemble, lock, preflight, read_json,
)
stage0 = backbone_config(DATA_DIR, BACKBONE_DIR, TRACK, SEED, DEVICE, WORKERS)
RUN_DIR = OUTPUT_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
# Check the selected backend before downloading data or fitting any model.
torch.empty(1, device=DEVICE)
print("Device:", DEVICE)
print("Shared backbone:", BACKBONE_DIR)
print("Raw features:", INPUT_DIR)
print("Experiment:", RUN_DIR)


## 2. Dataset preparation and validation

An existing dataset is used directly. Otherwise the repository preparation script downloads and extracts Waterbirds. The next check verifies official split counts, unique IDs, all four groups and every image path before the long training stage. It does not compute test predictions or metrics.

In [ ]:
metadata_candidates = [DATA_DIR / "metadata_waterbirds.csv", DATA_DIR / "metadata.csv",
                       DATA_DIR / "waterbird_complete95_forest2water2/metadata.csv"]
if not any(path.is_file() for path in metadata_candidates):
    if not DOWNLOAD_DATA:
        raise FileNotFoundError(f"No metadata under {DATA_DIR}; set DATA_DIR or enable DOWNLOAD_DATA.")
    if DATA_DIR.name != "waterbirds":
        raise ValueError("Automatic preparation expects a directory named waterbirds. Prepare custom layouts externally.")
    subprocess.run([sys.executable, str(REPO / "scripts/prepare_waterbirds.py"),
                    "--data_dir", str(DATA_DIR.parent)], cwd=REPO, check=True)
metadata_path, _ = validate_dataset(DATA_DIR)
print("Verified official counts and image paths:", metadata_path)


## 3. Shared ERM fine-tuning

The fixed recipe uses cross entropy, class-balanced sampling, SGD with learning rate 0.001 and momentum 0.9, batch size 128 and 100 epochs. Standard ERM uses deterministic preprocessing. ERM* adds random resized crops, photometric distortion and horizontal flips to training only. The shared trainer records validation metrics, but the checkpoint used below is always the final epoch. Completed matching runs are reused; `ckpt_best_wga.pt` is not used.

In [ ]:
# Uses ImageNet-pretrained ResNet-50 and fine-tunes all parameters for 100 epochs.
# No validation-based checkpoint is passed to H2; the final epoch is used.
checkpoint = train_or_reuse(stage0)
print("Final-epoch checkpoint:", checkpoint)


## 4. Feature extraction and experiment readiness

The shared model loader and feature extractor produce raw 2048-dimensional embeddings in metadata split order. The adapter writes `val_raw.npz`, `test_raw.npz`, a metadata copy and audited provenance automatically. It references the original checkpoint without duplicating it. The feature bundle is published atomically and checked on reuse. Both feature files are validated before head training.

In [ ]:
inputs = {key: Path(value) for key, value in export_inputs(stage0, INPUT_DIR).items()}
cfg = yaml.safe_load((REPO / f"configs/dpe_head_reweighting/waterbirds_{TRACK}.yaml").read_text())
cfg.update(track=TRACK, seed=SEED, device=DEVICE)
cfg.update({key: str(path) for key, path in inputs.items()})
cfg.update(experiment_dir=str(RUN_DIR / "experiment"),
           ensemble_path=str(RUN_DIR / "checkpoints/ensemble.pt"),
           results_dir=str(RUN_DIR / "results"))
CONFIG_PATH = RUN_DIR / "config.yaml"
resolved_yaml = yaml.safe_dump(cfg, sort_keys=False)
if CONFIG_PATH.exists() and CONFIG_PATH.read_text() != resolved_yaml:
    raise ValueError("This RUN_NAME already has a different configuration. Choose a new RUN_NAME.")
CONFIG_PATH.write_text(resolved_yaml)
resolved = config(CONFIG_PATH)
print("Readiness:", preflight(resolved))

def run(action):
    print(f"Running {action}: {TRACK}, seed {SEED}, {DEVICE}", flush=True)
    subprocess.run([sys.executable, str(REPO / "scripts/dpe_head_reweighting.py"),
                    action, "--config", str(CONFIG_PATH)], cwd=REPO, check=True)

print("Stage-0 provenance:")
print(inputs["provenance"].read_text())


## 5. Validation partition preparation

Preparation verifies artifact hashes and official split counts, then persists a group-stratified 60% A / 20% B / 20% C partition using split seed 0. Test features are not required at this stage.

In [ ]:
manifest_path = Path(resolved["experiment_dir"]) / "manifest.json"
if manifest_path.exists():
    context(resolved)  # Reject changed inputs instead of silently reusing them.
    print("Reusing verified manifest.")
else:
    run("prepare")
manifest = read_json(manifest_path)
print("Subset sizes:", {role: len(rows) for role, rows in manifest["parts"].items()})

## 6. Sequential head training

Fifteen heads are fitted using group-balanced samples from A, with 20 epochs per head. Epoch selection also uses A. Neither B, C nor test data is used for head fitting or epoch selection. The completed ensemble is exported for subsequent selection and evaluation.

In [ ]:
if Path(resolved["ensemble_path"]).exists():
    context(resolved)
    heads = get_ensemble(resolved)
    print("Reusing verified heads; selected epochs:", heads["selected_epochs"])
else:
    run("train-heads")

## 7. Reliability estimation and aggregation selection

Subset B determines per-head reliability scores and candidate weights. Subset C selects among uniform averaging and the 12 declared weighted configurations. The selection record contains all candidate metrics, selected weights and input identities. Uniform selection is retained as a valid outcome.

In [ ]:
selection_path = Path(resolved["experiment_dir"]) / "selection.json"
if selection_path.exists():
    context(resolved)
    selection = read_json(selection_path)
    if selection["lock"] != lock(resolved):
        raise ValueError("Saved selection inputs changed; use a fresh experiment.")
    print("Reusing frozen selection.")
else:
    run("fit-select")
selection = read_json(selection_path)
print(json.dumps(selection["selected"], indent=2))

## 8. Held-out evaluation

Evaluation requires a persisted selection record. The uniform control and selected aggregation are evaluated using identical test features and frozen heads. The predefined grid and partition remain fixed after test evaluation.

In [ ]:
metrics_path = Path(resolved["results_dir"]) / "metrics.json"
if metrics_path.exists():
    context(resolved)
    result = read_json(metrics_path)
    if result["lock"] != lock(resolved) or result["test_features_hash"] != digest(inputs["test_features"]) or result["metadata_hash"] != digest(inputs["metadata"]):
        raise ValueError("Existing test result inputs changed; preserve the original experiment.")
    print("Displaying existing final evaluation.")
else:
    if not inputs["test_features"].is_file():
        raise FileNotFoundError(f"Required test feature file is missing: {inputs['test_features']}")
    run("evaluate")
result = read_json(metrics_path)
for method in ("uniform", "selected"):
    m = result[method]
    print(f"{method:>8}: accuracy={m['overall']:.2%}, WGA={m['wga']:.2%}")
    print("          per-group accuracy:", m["groups"])
print("Paired changes (percentage points):", result["delta_pp"])
print("Effective head count:", result["effective_heads"])
from IPython.display import display, Image
plot = Path(resolved["results_dir"]) / "head_diagnostics.png"
display(Image(filename=str(plot)))

## 9. Artifact export

The results archive contains the resolved H2 and backbone configuration, shared backbone run metadata, training summary, handoff hashes, validation partition, trained heads, selection record, metrics, prediction caches and plots. Large backbone checkpoints and raw feature arrays remain in `BACKBONE_DIR` and `INPUT_DIR`; retain those directories separately when moving or archiving an experiment.

For a multi-seed study, repeat the notebook with seeds 0, 1 and 2 and report each track separately. The ZIP is written beside the run directory under `OUTPUT_DIR`.

In [ ]:
import platform
import shutil

# Record notebook execution metadata alongside the runner's artifact provenance.
environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": str(torch.__version__),
    "device": cfg["device"],
    "accelerator": torch.cuda.get_device_name(0) if cfg["device"] == "cuda" else "CPU",
}
(RUN_DIR / "notebook_environment.json").write_text(json.dumps(environment, indent=2) + "\n")
stage0_records = RUN_DIR / "stage0_records"
stage0_records.mkdir(exist_ok=True)
for name in ("config.yaml", "run.json", "summary.json"):
    shutil.copyfile(BACKBONE_DIR / name, stage0_records / name)
for name in ("provenance.json", "handoff.json"):
    shutil.copyfile(INPUT_DIR / name, stage0_records / name)
zip_path = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
print("Archive:", zip_path)